# 商家类别：大语言模型辅助分类

本 Notebook 只处理规则分类后仍为“**不确定**”的商家，不会重新处理已由关键词规则明确分类的记录。

为节省 token，处理单位是 **商家（restId）**，不是单条评论：每家最多选取 2 条信息量较高的评论，每批让模型判断 10 家商家；模型只返回类别和置信度，不生成长解释。

> 建议先保持 `RUN_LIMIT = 20` 测试。确认结果可靠后，再改为 `None` 处理全部任务。


## 第 1 步：配置路径、抽样和 token 控制参数

- `MAX_REVIEWS_PER_REST = 2`：每家最多给模型 2 条评论。
- `MAX_CHARS_PER_COMMENT = 120`：每条评论截断至 120 个字符。
- `BATCH_SIZE = 10`：10 家商家共用一段任务说明，降低重复提示词开销。
- `RUN_LIMIT = 20`：首次只调用 20 家，避免误消耗 token。


In [22]:
from pathlib import Path
import csv
import json
import re
from collections import Counter, defaultdict

PROJECT_DIR = Path(r"D:\携程实习文件\项目2\推荐系统数据集\推荐系统数据集\店家")
RAW_DIR = PROJECT_DIR / "data" / "raw"
DERIVED_DIR = PROJECT_DIR / "data" / "derived"
FINAL_DIR = PROJECT_DIR / "data" / "final"
RATING_SPLIT_DIR = PROJECT_DIR / "data" / "splits" / "rating_split"
NAME_OUTPUT_DIR = PROJECT_DIR / "outputs" / "name_completion"
CATEGORY_OUTPUT_DIR = PROJECT_DIR / "outputs" / "category"
CATEGORY_INPUT_PATH = DERIVED_DIR / "restaurants_category_draft.csv"
# RATING_SPLIT_DIR 已在首个代码单元格中定义。
TASK_PATH = CATEGORY_OUTPUT_DIR / "category_llm_tasks.csv"
LLM_RESULT_PATH = CATEGORY_OUTPUT_DIR / "category_llm_screened.csv"
FINAL_OUTPUT_PATH = FINAL_DIR / "restaurants_final.csv"

TARGET_CATEGORY = "不确定"
MAX_REVIEWS_PER_REST = 2
MAX_CHARS_PER_COMMENT = 120
BATCH_SIZE = 10  # 短 JSON 输出时可保持 10 家一批

# 第一次务必保持 20；检查结果后再改为 None。
RUN_LIMIT = None


## 第 2 步：读取规则分类结果，确定待处理商家

这里只选择当前“描述类别”为“不确定”且至少有一条非空评论的商家。


In [23]:
with CATEGORY_INPUT_PATH.open("r", encoding="utf-8-sig", newline="") as source:
    category_rows = list(csv.DictReader(source))

name_by_rest_id = {row["restId"]: (row.get("name") or "").strip() for row in category_rows}
target_rest_ids = {
    row["restId"] for row in category_rows
    if row.get("描述类别") == TARGET_CATEGORY
}

print(f"全部商家数：{len(category_rows):,}")
print(f"待大模型处理的“不确定”商家数：{len(target_rest_ids):,}")


全部商家数：243,247
待大模型处理的“不确定”商家数：21,162


## 第 3 步：按商家抽取高信息量评论并生成任务表

评论按长度和是否包含消费线索词进行排序；每家只保留前 2 条，避免把大量“服务不错”“环境一般”这类短信息重复发送给模型。

运行后会生成 `category_llm_tasks.csv`。此步骤不调用 API，也不消耗 token。


In [24]:
FOCUS_HINTS = (
    "吃", "喝", "菜", "餐", "饭", "面", "粉", "肉", "鱼", "火锅", "烧烤", "寿司", "咖啡", "奶茶",
    "超市", "便利店", "酒店", "宾馆", "酒吧", "影院", "KTV", "驾校", "医院", "银行", "公园", "商场",
)

def normalize_comment(text):
    return re.sub(r"\s+", " ", (text or "").strip())

def information_score(text):
    return min(len(text), MAX_CHARS_PER_COMMENT) + 35 * sum(hint in text for hint in FOCUS_HINTS)

selected_comments = defaultdict(list)
comment_count_by_rest = Counter()

for part_path in sorted(RATING_SPLIT_DIR.glob("ratings_part_*.csv")):
    with part_path.open("r", encoding="utf-8-sig", newline="") as source:
        for row in csv.DictReader(source):
            rest_id = row.get("restId")
            if rest_id not in target_rest_ids:
                continue
            comment = normalize_comment(row.get("comment"))
            if not comment:
                continue
            comment_count_by_rest[rest_id] += 1
            clipped = comment[:MAX_CHARS_PER_COMMENT]
            bucket = selected_comments[rest_id]
            bucket.append((information_score(clipped), clipped))
            bucket.sort(key=lambda item: item[0], reverse=True)
            del bucket[MAX_REVIEWS_PER_REST:]

tasks = []
for rest_id, comments in selected_comments.items():
    tasks.append({
        "restId": rest_id,
        "name": name_by_rest_id.get(rest_id, ""),
        "comment_count": comment_count_by_rest[rest_id],
        "selected_comment_count": len(comments),
        "comments": "\n".join(f"评论{index + 1}：{text}" for index, (_, text) in enumerate(comments)),
    })

tasks.sort(key=lambda row: (-row["comment_count"], row["restId"]))
with TASK_PATH.open("w", encoding="utf-8-sig", newline="") as target:
    writer = csv.DictWriter(target, fieldnames=["restId", "name", "comment_count", "selected_comment_count", "comments"])
    writer.writeheader()
    writer.writerows(tasks)

selected_chars = sum(len(row["comments"]) for row in tasks)
print(f"生成任务数：{len(tasks):,}")
print(f"发送给模型的评论字符总量：{selected_chars:,}")
print(f"粗略输入 token 量（仅评论正文）：约 {selected_chars * 0.8:,.0f}")
print(f"已保存：{TASK_PATH.name}")
print("前 3 条任务：")
for row in tasks[:3]:
    print(row)


生成任务数：20,035
发送给模型的评论字符总量：2,368,986
粗略输入 token 量（仅评论正文）：约 1,895,189
已保存：category_llm_tasks.csv
前 3 条任务：
{'restId': '74135', 'name': '', 'comment_count': 100, 'selected_comment_count': 2, 'comments': '评论1：页面很清桑，查询结果也比较准确，一般来说还是挺喜欢用这个在出行之前查询线路的。但是不知道怎么回事，很多时候着急用就是打不开，一开始觉得是电脑的问题，但是后来换了电脑还是不行，而且遇到过不止一两次，所以还是觉得是网站不够稳定的原因...希望加\n评论2：北京公交网是北京公交公司的网站。在网站上查询公交线路非常的方便。在网站上输入起始地和终点站，就可以查询到可以乘坐的所有车次。除了查询乘车路线，网站上还有以交新闻、充值点信息、旅游景点服务等内容，非常的实用。 另外公交网的页面设计也很特别，网'}
{'restId': '9260', 'name': '', 'comment_count': 54, 'selected_comment_count': 2, 'comments': '评论1：轻轨建的不错，站台修的也漂亮，真有点到日本的感觉，哈哈哈。 有一点需要注意，他的票是要回收的，所以需要报销的同学请您给售票处要一张报销票，要不然你还真找不到票替代。 车辆里面和地铁差不多，干净，有提醒设备，唯一不同的就是，做这个车有周边的风\n评论2：在塘沽开发区上班，所以经常乘坐。 车速很快，早高峰时发车频率也很快，干净，有空调很舒适。 可以刷城市卡，也可以充值，很方便。 只是原来中山门是起点站时，车刚进站，大家一拥而上抢座的场面比较壮观。 当然，我也是其中一员。。。'}
{'restId': '220393', 'name': '', 'comment_count': 43, 'selected_comment_count': 2, 'comments': '评论1：就是喜欢坐在2层驾驶员的位置，看看中环线的风景，做一圈的机会没有了，现在忙了，多看看天津的变化对自己也是一个鞭策，哈哈，成说教了，印象中47和48 的下面那一层很矮，所以一上车就往二层跑，尽量找比较靠前的位子坐，唉，新

## 第 4 步：安装客户端并安全输入 DeepSeek API Key

只需首次运行安装单元格。API Key 通过隐藏输入读取，只保留在当前 Notebook 内存中，不会写入 CSV 或 Notebook 文件。

当前代码使用 DeepSeek 的 OpenAI 兼容接口、JSON 输出和非思考模式；分类任务不需要长推理。


In [25]:
!pip install -q -U openai


In [26]:
import getpass
from openai import OpenAI

MODEL_NAME = "deepseek-v4-flash"  # 若账户没有该模型，可改成 DeepSeek 控制台显示的可用模型名。
api_key = getpass.getpass("请输入 DeepSeek API Key（输入不会显示）：")
client = OpenAI(api_key=api_key, base_url="https://api.deepseek.com")

print(f"已配置模型：{MODEL_NAME}")


已配置模型：deepseek-v4-flash


## 第 5 步：定义低 token 的批量分类函数

模型只能从既定类别中选择。它只返回：`restId`、`描述类别`、`置信度`和最多 8 个字的依据；若证据不足必须返回“不确定”。


In [27]:
CATEGORY_LABELS = [
    "电商平台/线上零售", "超市/便利店", "市场/花鸟市场", "景点/公园/公共场所", "教育培训/驾校",
    "餐饮-火锅烧烤", "餐饮-茶饮咖啡甜品", "酒吧/夜生活", "酒店/住宿", "购物零售",
    "生活服务", "休闲娱乐", "餐饮-正餐", "不确定",
]
ALLOWED_LABELS = set(CATEGORY_LABELS)
SYSTEM_PROMPT = "你是商家类别标注助手。仅依据名称和评论分类；证据不足时选不确定。必须只输出 JSON。"

def classify_batch(batch, max_retries=3):
    payload = [
        {"restId": row["restId"], "name": row["name"], "comments": row["comments"]}
        for row in batch
    ]
    prompt = f"""从下列类别中为每个商家选择一个类别：{"、".join(CATEGORY_LABELS)}。
仅根据名称和评论判断；仅有服务、环境等泛化描述时选择“不确定”。
只输出 JSON，不能有任何解释、Markdown 或额外字段。格式必须严格为：
{{"results":[{{"restId":"", "描述类别":"", "置信度":"高/中/低"}}]}}
待分类数据：
{json.dumps(payload, ensure_ascii=False)}
"""
    expected_ids = {str(row["restId"]) for row in batch}
    last_error = None
    for attempt in range(1, max_retries + 1):
        try:
            response = client.chat.completions.create(
                model=MODEL_NAME,
                messages=[
                    {"role": "system", "content": SYSTEM_PROMPT},
                    {"role": "user", "content": prompt},
                ],
                response_format={"type": "json_object"},
                max_tokens=350,
                extra_body={"thinking": {"type": "disabled"}},
            )
            results = json.loads(response.choices[0].message.content).get("results", [])
            returned_ids = {str(item.get("restId", "")) for item in results}
            if len(results) != len(batch) or returned_ids != expected_ids:
                raise ValueError(f"模型返回 {len(results)} 条，预期 {len(batch)} 条")
            for item in results:
                if item.get("描述类别") not in ALLOWED_LABELS:
                    item["描述类别"] = "不确定"
                if item.get("置信度") not in {"高", "中", "低"}:
                    item["置信度"] = "低"
                item["依据"] = ""  # 为省 token，不要求模型生成依据。
            return results
        except Exception as error:
            last_error = error
            print(f"本批第 {attempt}/{max_retries} 次尝试失败：{error}")
            time.sleep(attempt)
    raise last_error


## 第 6 步：分批调用、断点续跑并保存模型结果

每完成一批即写入 `category_llm_screened.csv`。若中途停止，再运行本单元格会自动跳过已完成的 `restId`。

默认只跑 20 家；检查 CSV 后再将第 1 步的 `RUN_LIMIT` 改为 `None`。


In [31]:
import time

with TASK_PATH.open("r", encoding="utf-8-sig", newline="") as source:
    all_tasks = list(csv.DictReader(source))

finished_ids = set()
if LLM_RESULT_PATH.exists():
    with LLM_RESULT_PATH.open("r", encoding="utf-8-sig", newline="") as source:
        finished_ids = {row["restId"] for row in csv.DictReader(source)}

pending = [row for row in all_tasks if row["restId"] not in finished_ids]
if RUN_LIMIT is not None:
    pending = pending[:RUN_LIMIT]

print(f"全部任务：{len(all_tasks):,}；已完成：{len(finished_ids):,}；本次待处理：{len(pending):,}")

output_fields = ["restId", "name", "comment_count", "selected_comment_count", "comments", "描述类别", "置信度", "依据"]
write_header = not LLM_RESULT_PATH.exists()

for start in range(0, len(pending), BATCH_SIZE):
    batch = pending[start:start + BATCH_SIZE]
    try:
        results = classify_batch(batch)
    except Exception as error:
        print(f"第 {start // BATCH_SIZE + 1} 批失败：{error}")
        print("此前结果已保存；检查后重新运行本单元格即可续跑。")
        break

    result_by_id = {item["restId"]: item for item in results}
    with LLM_RESULT_PATH.open("a", encoding="utf-8-sig", newline="") as target:
        writer = csv.DictWriter(target, fieldnames=output_fields)
        if write_header:
            writer.writeheader()
            write_header = False
        for row in batch:
            result = result_by_id[row["restId"]]
            # 模型偶尔会附加“注意”等额外字段；只写入预先约定的列，避免 CSV 报错。
            writer.writerow({
                **row,
                "描述类别": result.get("描述类别", "不确定"),
                "置信度": result.get("置信度", "低"),
                "依据": result.get("依据", ""),
            })

    print(f"已完成：{min(start + len(batch), len(pending)):,}/{len(pending):,}")
    time.sleep(0.2)


全部任务：20,035；已完成：7,283；本次待处理：12,752
已完成：10/12,752
已完成：20/12,752
已完成：30/12,752
已完成：40/12,752
已完成：50/12,752
已完成：60/12,752
已完成：70/12,752
已完成：80/12,752
已完成：90/12,752
已完成：100/12,752
已完成：110/12,752
已完成：120/12,752
已完成：130/12,752
已完成：140/12,752
已完成：150/12,752
已完成：160/12,752
已完成：170/12,752
已完成：180/12,752
已完成：190/12,752
已完成：200/12,752
已完成：210/12,752
已完成：220/12,752
已完成：230/12,752
已完成：240/12,752
已完成：250/12,752
已完成：260/12,752
已完成：270/12,752
已完成：280/12,752
已完成：290/12,752
已完成：300/12,752
已完成：310/12,752
已完成：320/12,752
已完成：330/12,752
已完成：340/12,752
已完成：350/12,752
已完成：360/12,752
已完成：370/12,752
已完成：380/12,752
已完成：390/12,752
已完成：400/12,752
已完成：410/12,752
已完成：420/12,752
已完成：430/12,752
已完成：440/12,752
已完成：450/12,752
已完成：460/12,752
已完成：470/12,752
已完成：480/12,752
已完成：490/12,752
已完成：500/12,752
已完成：510/12,752
已完成：520/12,752
已完成：530/12,752
已完成：540/12,752
已完成：550/12,752
已完成：560/12,752
已完成：570/12,752
已完成：580/12,752
已完成：590/12,752
已完成：600/12,752
已完成：610/12,752
已完成：620/12,752
已完成：630/12,752
已完成：640/12,752
已完成：650/12,752

## 第 7 步：检查模型结果

建议先人工查看高、中、低置信度样本，再决定是否写回最终类别。


In [ ]:
with LLM_RESULT_PATH.open("r", encoding="utf-8-sig", newline="") as source:
    llm_rows = list(csv.DictReader(source))

print(f"已完成模型分类：{len(llm_rows):,}")
print("类别统计：", Counter(row["描述类别"] for row in llm_rows))
print("置信度统计：", Counter(row["置信度"] for row in llm_rows))

for row in llm_rows[:10]:
    print({key: row[key] for key in ["restId", "name", "描述类别", "置信度", "依据", "comments"]})


已完成模型分类：20
类别统计： Counter({'不确定': 7, '购物零售': 5, '休闲娱乐': 3, '生活服务': 3, '教育培训/驾校': 1, '景点/公园/公共场所': 1})
置信度统计： Counter({'中': 10, '高': 10})
{'restId': '74135', 'name': '', '描述类别': '不确定', '置信度': '中', '依据': '评论内容为公交查询网站，无法明确归类', 'comments': '评论1：页面很清桑，查询结果也比较准确，一般来说还是挺喜欢用这个在出行之前查询线路的。但是不知道怎么回事，很多时候着急用就是打不开，一开始觉得是电脑的问题，但是后来换了电脑还是不行，而且遇到过不止一两次，所以还是觉得是网站不够稳定的原因...希望加\n评论2：北京公交网是北京公交公司的网站。在网站上查询公交线路非常的方便。在网站上输入起始地和终点站，就可以查询到可以乘坐的所有车次。除了查询乘车路线，网站上还有以交新闻、充值点信息、旅游景点服务等内容，非常的实用。 另外公交网的页面设计也很特别，网'}
{'restId': '9260', 'name': '', '描述类别': '不确定', '置信度': '中', '依据': '评论为轻轨设施体验', 'comments': '评论1：轻轨建的不错，站台修的也漂亮，真有点到日本的感觉，哈哈哈。 有一点需要注意，他的票是要回收的，所以需要报销的同学请您给售票处要一张报销票，要不然你还真找不到票替代。 车辆里面和地铁差不多，干净，有提醒设备，唯一不同的就是，做这个车有周边的风\n评论2：在塘沽开发区上班，所以经常乘坐。 车速很快，早高峰时发车频率也很快，干净，有空调很舒适。 可以刷城市卡，也可以充值，很方便。 只是原来中山门是起点站时，车刚进站，大家一拥而上抢座的场面比较壮观。 当然，我也是其中一员。。。'}
{'restId': '220393', 'name': '', '描述类别': '不确定', '置信度': '中', '依据': '评论为公交巴士乘坐体验', 'comments': '评论1：就是喜欢坐在2层驾驶员的位置，看看中环线的风景，做一圈的机会没有了，现在忙了，多看看天津的变化对自己也是一个鞭策，哈哈，成说教了，印象中47和48 的下面那一

## 第 8 步：只写回高置信度结果，生成新文件

原来的 `restaurants_category_draft.csv` 不会被覆盖。新文件仅将模型“高置信度”且非“不确定”的结果写回；其余商家保持原分类。


In [ ]:
ACCEPTED_CONFIDENCE = {"高"}

accepted = {
    row["restId"]: row["描述类别"]
    for row in llm_rows
    if row["置信度"] in ACCEPTED_CONFIDENCE and row["描述类别"] != "不确定"
}

merged_rows = []
for row in category_rows:
    merged_rows.append({
        "restId": row["restId"],
        "name": row.get("name", ""),
        "描述类别": accepted.get(row["restId"], row["描述类别"]),
    })

with FINAL_OUTPUT_PATH.open("w", encoding="utf-8-sig", newline="") as target:
    writer = csv.DictWriter(target, fieldnames=["restId", "name", "描述类别"])
    writer.writeheader()
    writer.writerows(merged_rows)

print(f"被模型高置信度补充的商家数：{len(accepted):,}")
print(f"已生成：{FINAL_OUTPUT_PATH.name}")
print(Counter(row["描述类别"] for row in merged_rows))


被模型高置信度补充的商家数：10
已生成：restaurants_category_llm_draft.csv
Counter({'餐饮-正餐': 124803, '餐饮-茶饮咖啡甜品': 30922, '餐饮-火锅烧烤': 23884, '不确定': 21152, '酒店/住宿': 9053, '购物零售': 8462, '景点/公园/公共场所': 5239, '生活服务': 5219, '休闲娱乐': 4648, '超市/便利店': 4353, '酒吧/夜生活': 3419, '教育培训/驾校': 1094, '电商平台/线上零售': 657, '市场/花鸟市场': 342})


## 报告可用说明

> 本研究先使用关键词规则对商家进行初步类别识别，再仅对规则无法可靠分类的商家实施大语言模型辅助判断。为降低成本与减少冗余信息，以商家为单位聚合评论，每家选取不超过两条信息量较高的评论；模型使用固定类别集合和 JSON 结构化输出。仅将高置信度且非“不确定”的结果写入新的结果文件，原始规则分类文件始终保留。
